#### This file was used to write the data to the database

In [1]:
import json
with open("data.json") as data:
    d = json.load(data)

In [14]:
# from pprint import pprint
import math
for k, v in d.items():

    f_coef = d[k][0]["TransFullCoef"]

    if math.isnan(f_coef):
        print("nan")

    # for entry in d[k]:
    #     if entry["TransFullCoef"] != f_coef and not math.isnan(entry["TransFullCoef"]):
    #         print(k)
    #         print("Different!")
    #         print(entry["TransFullCoef"])
    #         print(f_coef)
    #         break


In [79]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="localhost",
        database="postgres",
        user="postgres",
        password="11111",
        port="5432"
    )

    # If connection is successful, you can now interact with the database
    print("Connection to PostgreSQL successful!")

except psycopg2.Error as e:
    print(f"Error connecting to PostgreSQL: {e}")


Connection to PostgreSQL successful!


In [78]:
conn.close()

In [80]:
with open("schema.sql") as f:
    schema_sql = f.read()
    tables = schema_sql.split(";")


In [81]:
cur = conn.cursor()

for table in tables:
    if table.strip():
        cur.execute(table)
conn.commit()

In [82]:
import math
cur = conn.cursor()
for k, v in d.items():

    f_coef = d[k][0]["TransFullCoef"]

    # contour_id = k
    # energy_export = d[k][0]["Active Energy Export (3:1-0:2.8.0*255:2)"]
    # energy_import = d[k][0]["Active Energy Import (3:1-0:1.8.0*255:2)"]
    # clock = d[k][0]["Clock (8:0-0:1.0.0*255:2)"]

    # print(f"{contour_id=}", f"{energy_export=}", f"{energy_import=}", f"{clock=}", f"{f_coef=}", sep="\n")
    contour_id = k

    cur.execute(
        """
        INSERT INTO contour (contour_id, fuel_coefficient)
        VALUES (%s, %s);
        """, (contour_id, f_coef))

    for entry in d[k]:
        energy_export = entry["Active Energy Export (3:1-0:2.8.0*255:2)"] if not math.isnan(entry["Active Energy Export (3:1-0:2.8.0*255:2)"]) else -1
        energy_import = entry["Active Energy Import (3:1-0:1.8.0*255:2)"] if not math.isnan(entry["Active Energy Import (3:1-0:1.8.0*255:2)"]) else -1
        clock = entry["Clock (8:0-0:1.0.0*255:2)"]
        try:
            cur.execute(
            """
            INSERT INTO contour_data (contour_id, energy_export, energy_import, clock)
            VALUES (%s, %s, %s, %s);
            """, (contour_id, energy_export, energy_import, clock))
        except Exception as e:
            print(f"Error inserting contour_data: {e}")
            print(f"{contour_id=}", f"{energy_export=}", f"{energy_import=}", f"{clock=}", sep="\n")
conn.commit()


